## ParTES Fig.2-style plotting (histogram + quantile function)

This notebook loads one ParTES walk-based experiment folder and reproduces Fig.2-style plots from:
- 《并行计时偏差评测指标及工具》(计算机科学, 2025) Fig.2

### Expected folder layout

Typically produced by `TacVar/scripts/run_partes_walk_normal.sh`:

```
<EXPR_ROOT>/
  walk_list_normal.csv
  meta.md            (optional)
  walks/
    w0000_taXXXXX/
      partes_ta_r0.csv ... partes_ta_r{np-1}.csv
      partes_tb_r0.csv ... (optional)
      partes_ta_cdf.csv, partes_tb_cdf.csv
      run.log
```

Timer count note: your experiment folder may represent only **one** timer; if you have multiple timers, point the notebook at each folder and compare the plots.

In [ ]:
from __future__ import annotations

import csv
import math
import os
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

# === Set this to your timestamp experiment folder ===
# Examples:
#   Path("/astrum/home/hpchzy/code/TacVar/scripts/output/20260311_140141")
#   Path("/astrum/home/hpchzy/code/data/20260311/20260311_140405")
EXPR_ROOT = Path("/astrum/home/hpchzy/code/data/20260311/20260311_140405")

# Which gauge to plot from ParTES output. Usually 'ta' is enough.
GAUGE_SIDE = "ta"  # 'ta' or 'tb'

# Histogram bins (None => numpy auto)
HIST_BINS = 80

# Quantile resolution for the ICDF plot
N_QUANTILES = 1000

EXPR_ROOT

In [ ]:
@dataclass
class Meta:
    expr_name: str = ""
    expr_id: str = ""
    timer: str = ""
    gauge: str = ""
    fkern: str = ""
    fsize_kib: str = ""
    rkern: str = ""
    rsize_kib: str = ""
    np: Optional[int] = None
    ntests: Optional[int] = None
    nwalks: Optional[int] = None


def _read_text_if_exists(p: Path) -> str:
    try:
        return p.read_text(encoding="utf-8", errors="replace")
    except FileNotFoundError:
        return ""


def parse_meta(expr_root: Path) -> Meta:
    """Parse meta.md produced by the driver script (optional)."""
    md = _read_text_if_exists(expr_root / "meta.md")
    m = Meta()
    if not md.strip():
        return m

    def grab(key: str) -> str:
        # meta.md table rows look like: | key | `value` |
        pat = re.compile(rf"\|\s*{re.escape(key)}\s*\|\s*`([^`]*)`\s*\|", re.IGNORECASE)
        mm = pat.search(md)
        return mm.group(1).strip() if mm else ""

    m.expr_name = grab("expr_name")
    m.expr_id = grab("expr_id")
    m.timer = grab("timer")
    m.gauge = grab("gauge")
    m.fkern = grab("fkern")
    m.fsize_kib = grab("fsize (KiB)")
    m.rkern = grab("rkern")
    m.rsize_kib = grab("rsize (KiB)")

    np_s = grab("np")
    if np_s.isdigit():
        m.np = int(np_s)
    ntests_s = grab("ntests")
    if ntests_s.isdigit():
        m.ntests = int(ntests_s)
    nwalks_s = grab("nwalks")
    if nwalks_s.isdigit():
        m.nwalks = int(nwalks_s)

    return m


def load_walk_list(expr_root: Path) -> pd.DataFrame:
    # Accept either walk_list_normal.csv or any walk_list_*.csv
    candidates = [expr_root / "walk_list_normal.csv"]
    candidates += sorted(expr_root.glob("walk_list_*.csv"))
    for p in candidates:
        if p.exists():
            df = pd.read_csv(p)
            if {"walk_idx", "ta_ns"}.issubset(df.columns):
                return df
    raise FileNotFoundError(f"No walk list found under {expr_root}")


def iter_measurement_csvs(expr_root: Path, side: str) -> list[Path]:
    # Walk directories named like wXXXX_taYYYYY
    walks_dir = expr_root / "walks"
    if not walks_dir.exists():
        raise FileNotFoundError(f"Missing walks/ under {expr_root}")

    # Example: partes_ta_r0.csv
    paths = sorted(walks_dir.glob(f"w*_ta*/partes_{side}_r*.csv"))
    if not paths:
        raise FileNotFoundError(f"No partes_{side}_r*.csv found under {walks_dir}")
    return paths


def load_int64_column(path: Path) -> np.ndarray:
    # Fast load: each CSV is a single int per line.
    # Use pandas for robustness against stray spaces.
    s = pd.read_csv(path, header=None, dtype=np.int64)[0]
    return s.to_numpy()


def compute_icdf(samples: np.ndarray, n_quantiles: int = 1000) -> tuple[np.ndarray, np.ndarray]:
    samples = np.asarray(samples, dtype=np.float64)
    samples = samples[np.isfinite(samples)]
    if samples.size == 0:
        return np.array([]), np.array([])
    qs = np.linspace(0.0, 1.0, n_quantiles)
    vals = np.quantile(samples, qs, method="linear")
    return qs, vals


meta = parse_meta(EXPR_ROOT)
walk_df = load_walk_list(EXPR_ROOT)
csvs = iter_measurement_csvs(EXPR_ROOT, GAUGE_SIDE)

meta, walk_df.head(), len(csvs)

In [ ]:
# Theoretical distribution (dashed curve in Fig.2): use ta_ns from walk list.
# In the paper, this corresponds to the configured base-time distribution.
theory = walk_df["ta_ns"].to_numpy(dtype=np.float64)

# Measured distribution (solid curve in Fig.2): aggregate all per-rank per-walk measurements.
# Warning: this can be large (nwalks * np * ntests).
chunks = []
for p in csvs:
    chunks.append(load_int64_column(p))
meas = np.concatenate(chunks).astype(np.float64)

len(theory), len(meas), meas.min(), meas.mean(), meas.max()

In [ ]:
# Plot Fig.2 style: histogram (left) and quantile function (right)
qs_t, icdf_t = compute_icdf(theory, n_quantiles=N_QUANTILES)
qs_m, icdf_m = compute_icdf(meas, n_quantiles=N_QUANTILES)

title_left = f"{meta.timer or 'timer?'} / histogram ({GAUGE_SIDE})"
title_right = f"{meta.timer or 'timer?'} / quantile function ({GAUGE_SIDE})"

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

# Histogram
ax = axes[0]
ax.hist(meas, bins=HIST_BINS, color="#4C78A8", alpha=0.85)
ax.set_title(title_left)
ax.set_xlabel("measured time (ns)")
ax.set_ylabel("count")

# Quantile function
ax = axes[1]
ax.plot(qs_m, icdf_m, color="#F58518", lw=2.0, label="measured")
ax.plot(qs_t, icdf_t, color="#54A24B", lw=2.0, ls="--", label="theoretical")
ax.set_title(title_right)
ax.set_xlabel("cumulative probability (quantile)")
ax.set_ylabel("time (ns)")
ax.legend(loc="best")

subtitle = (
    f"expr={meta.expr_name or 'n/a'}"
    + (f" id={meta.expr_id}" if meta.expr_id else "")
    + f" | gauge={meta.gauge or 'n/a'}"
    + f" | fkern={meta.fkern or 'n/a'} fsize={meta.fsize_kib or 'n/a'}KiB"
    + f" | np={meta.np or 'n/a'}"
)
fig.suptitle(subtitle, y=1.03, fontsize=10)
fig.tight_layout()
plt.show()

### Notes / extensions

- If you want to reproduce Fig.2 for **multiple timers**, run the same experiment multiple times (one timer per folder) and then load/plot each folder in a loop.
- If your output folder contains multiple experiment roots (each with its own `meta.md` and `walks/`), you can extend this notebook to auto-discover them and create a multi-row plot grid (one row per timer).
